# Wikipedia Category-Based Subset Extraction

- Locates the project root by walking up the directory tree until a `DATA` folder is found.
- Defines a `normalize()` helper that lowercases titles, replaces spaces/underscores, strips accents, parentheses and category prefixes.
- Crawls the Wikipedia API (`list=categorymembers`) recursively over a set of categories (with a configurable depth) to collect all article titles.
- Matches the normalized crawled titles against the MCQ evaluation CSV (`title` column) to build one thematic subset per category, saved under `DATA/SUBSETS_<LANG>/ARTICLES_SUBSETS_<LANG>/`.
- Post-processes all subsets to make them mutually disjoint (each `title_norm` is assigned to the smallest dataset), then reports per-dataset and global statistics on overlap removal and `content` character counts.

In [ ]:
from pathlib import Path

# Walk up the tree until the DATA folder is found
ROOT = Path().resolve()
while not (ROOT / "DATA").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DATA = ROOT / "DATA"
print("CLEAN root:", ROOT)

In [ ]:
import pandas as pd 
df = pd.read_csv(str(DATA / "QUESTIONS" / "mcq_pt.csv"))


In [ ]:
import unicodedata
import re

def normalize(title):
    # 1. Convert to lowercase
    title = title.lower()

    # 2. Replace spaces and underscores by a single underscore
    title = re.sub(r'[\s_]+', '_', title)

    # 3. Remove accents and special characters
    title = unicodedata.normalize('NFKD', title)
    title = ''.join(c for c in title if not unicodedata.combining(c))

    # 4. Remove parentheses and their content (e.g. "Pizza (dish)" -> "pizza")
    title = re.sub(r'\(.*\)', '', title)

    # 5. Remove prefixes such as "categoría:" or "catégorie:"
    title = re.sub(r'^categor[aí]a_', '', title, flags=re.IGNORECASE)

    # 6. Remove leftover empty separators
    title = title.strip('_')

    return title

## Spanish (ES) subset extraction

In [ ]:
import requests
import time

HEADERS = {
    "User-Agent": "MyBot/1.0 (https://example.com/bot-info)"
}

def get_category_members(category, depth=2, lang="es", visited=None):
    if visited is None:
        visited = set()
    # Stop recursion on already-seen categories or when max depth is reached
    if category in visited or depth < 0:
        return set()
    visited.add(category)

    api = f"https://{lang}.wikipedia.org/w/api.php"
    titles = set()
    subcats = []
    cmcontinue = None

    while True:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": category,
            "cmlimit": "500",
            "cmtype": "page|subcat",
            "format": "json",
            "formatversion": "2",
        }
        if cmcontinue:
            params["cmcontinue"] = cmcontinue

        resp = requests.get(api, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()                # <- will surface the real HTTP error
        try:
            r = resp.json()
        except Exception:
            print("Non-JSON response received:")
            print(resp.text[:500])             # debug
            raise

        # ns == 14 -> subcategory, ns == 0 -> article
        for m in r.get("query", {}).get("categorymembers", []):
            if m["ns"] == 14:
                subcats.append(m["title"])
            elif m["ns"] == 0:
                titles.add(m["title"])

        if "continue" in r:
            cmcontinue = r["continue"]["cmcontinue"]
        else:
            break
        time.sleep(0.01)

    # Recurse into subcategories
    for sub in subcats:
        titles |= get_category_members(sub, depth=depth-1, lang=lang, visited=visited)
        time.sleep(0.01)

    return titles




categories_arts = [
    "Categoría:Literatura en Argentina",
    "Categoría:Literatura en Bolivia",
    "Categoría:Literatura en Chile",
    "Categoría:Literatura en Colombia",
    "Categoría:Literatura en Ecuador",
    "Categoría:Literatura en México",
    "Categoría:Literatura en Paraguay",
    "Categoría:Literatura en Perú",
    "Categoría:Literatura en Uruguay",
    "Categoría:Literatura en Venezuela"
    # Variants:
]

all_titles = {}
for cat in categories_arts:
    print(f"-> {cat}")
    all_titles[cat] = get_category_members(cat, depth=2)
    print(f"   {len(all_titles[cat])} articles")

In [ ]:
# Global union of all crawled titles
union_titles = set().union(*all_titles.values())
union_norm = {normalize(t) for t in union_titles}

import pandas as pd 

df = pd.read_csv(str(DATA / "QUESTIONS" / "mcq_eval_results_es__ministral-small.csv"))
print(df["title"])

# Normalize dataset titles to allow matching with the crawled set
df["title_norm"] = df["title"].astype(str).apply(normalize)
print(union_titles)
print(union_norm)

df_corpus = df[df["title_norm"].isin(union_norm)].copy()
print(f"\n✅ Final corpus: {len(df_corpus)} articles")

out_es = Path(str(DATA / "SUBSETS_ES" / "ARTICLES_SUBSETS_ES" / "artesania_articles_es.csv"))
out_es.parent.mkdir(parents=True, exist_ok=True)
df_corpus.to_csv(out_es, index=False)

## Making the datasets disjoint (for GNN and downstream tasks)

Each `title_norm` is assigned to a single dataset (the smallest one, ties broken by filename)
so that the thematic subsets no longer overlap. Detailed statistics are printed and saved.

In [ ]:
import glob
import os
import pandas as pd

# Dataset directory
DATA_DIR = str(DATA / "SUBSETS_ES" / "ARTICLES_SUBSETS_ES")
PATTERN = os.path.join(DATA_DIR, "*_articles_es.csv")

# 1. Automatic file detection
files = sorted(glob.glob(PATTERN))
files = [f for f in files if not f.endswith("_disjoint.csv")]
print(f"{len(files)} files detected\n")

# 2. Loading + internal deduplication on title_norm
datasets = {}          # path -> deduplicated DataFrame
raw_sizes = {}         # path -> raw size (before internal dedup)
orig_sizes = {}        # path -> size after internal dedup
internal_dups = {}     # path -> number of internal duplicates removed

for f in files:
    df = pd.read_csv(f, sep=",", encoding="utf-8")
    if "title_norm" not in df.columns:
        raise ValueError(f"Column 'title_norm' missing in {f}")
    raw_sizes[f] = len(df)
    df = df.drop_duplicates(subset="title_norm", keep="first").reset_index(drop=True)
    datasets[f] = df
    orig_sizes[f] = len(df)
    internal_dups[f] = raw_sizes[f] - orig_sizes[f]

# 3. Assign each title_norm to the smallest dataset
def sort_key(path):
    return (orig_sizes[path], os.path.basename(path))

winner = {}
for f in files:
    for tn in datasets[f]["title_norm"]:
        if tn not in winner or sort_key(f) < sort_key(winner[tn]):
            winner[tn] = f

# 4. Filtering + writing + stats collection
stats = []
final_dfs = {}

for f in files:
    df = datasets[f]
    # Keep only the rows this dataset "won"
    mask = df["title_norm"].map(lambda tn: winner[tn] == f)
    df_out = df[mask].reset_index(drop=True)
    final_dfs[f] = df_out

    base, ext = os.path.splitext(f)
    out_path = f"{base}_disjoint{ext}"
    df_out.to_csv(out_path, sep=",", encoding="utf-8", index=False)

    cat = os.path.basename(f).replace("_articles_es.csv", "")
    n_raw = raw_sizes[f]
    n_dedup = orig_sizes[f]
    n_final = len(df_out)

    stats.append({
        "categorie": cat,
        "brut": n_raw,
        "doublons_internes": internal_dups[f],
        "apres_dedup": n_dedup,
        "supprimes_chevauchement": n_dedup - n_final,
        "final": n_final,
        "pct_supprime": round(100 * (n_dedup - n_final) / n_dedup, 2) if n_dedup else 0.0,
    })

stats_df = pd.DataFrame(stats).sort_values("brut", ascending=False).reset_index(drop=True)

# 5. Display the per-dataset table
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)

print("=" * 100)
print("PER-DATASET STATISTICS")
print("=" * 100)
print(stats_df.to_string(index=False))

# 6. Global totals
total_raw = stats_df["brut"].sum()
total_internal = stats_df["doublons_internes"].sum()
total_dedup = stats_df["apres_dedup"].sum()
total_removed = stats_df["supprimes_chevauchement"].sum()
total_final = stats_df["final"].sum()

# Number of globally unique title_norm (= expected number of final rows)
unique_titles = len(winner)

print("\n" + "=" * 100)
print("GLOBAL STATISTICS")
print("=" * 100)
print(f"Number of datasets                          : {len(files)}")
print(f"Raw articles (before any processing)        : {total_raw}")
print(f"Internal duplicates removed                 : {total_internal}")
print(f"Articles after internal dedup               : {total_dedup}")
print(f"Articles removed (inter-dataset overlap)    : {total_removed}")
print(f"Final articles (total)                      : {total_final}")
print(f"Globally unique titles (sanity check)       : {unique_titles}")
print(f"  -> consistent: {total_final == unique_titles}")
print(f"Total reduction vs raw                      : "
      f"{round(100 * (total_raw - total_final) / total_raw, 2)}%")

# 7. Disjointness check across output files
print("\n" + "=" * 100)
print("DISJOINTNESS CHECK")
print("=" * 100)
seen = {}
overlap_found = False
for f in files:
    cat = os.path.basename(f).replace("_articles_es.csv", "")
    for tn in final_dfs[f]["title_norm"]:
        if tn in seen:
            overlap_found = True
            print(f"  OVERLAP: '{tn}' in {seen[tn]} and {cat}")
        else:
            seen[tn] = cat
if not overlap_found:
    print("  OK: no title_norm present in more than one output file.")

# 8. Save the statistics report to CSV
report_path = os.path.join(DATA_DIR, "disjoint_stats_report.csv")
stats_df.to_csv(report_path, sep=",", encoding="utf-8", index=False)
print(f"\nStats report saved: {report_path}")

print("\nDone.")

## Content size statistics on the disjoint datasets

In [ ]:
import glob
import os
import pandas as pd

DATA_DIR = str(DATA / "SUBSETS_ES" / "ARTICLES_SUBSETS_ES")

# Work on the disjoint files generated previously
disjoint_files = sorted(glob.glob(os.path.join(DATA_DIR, "*_articles_es_disjoint.csv")))
print(f"{len(disjoint_files)} disjoint files detected\n")

char_stats = []

for f in disjoint_files:
    df = pd.read_csv(f, sep=",", encoding="utf-8")
    cat = os.path.basename(f).replace("_articles_es_disjoint.csv", "")

    if "content" not in df.columns:
        raise ValueError(f"Column 'content' missing in {f}")

    # Character length of each cell (NaN -> 0)
    lengths = df["content"].fillna("").astype(str).str.len()

    char_stats.append({
        "categorie": cat,
        "n_lignes": len(df),
        "total_chars": int(lengths.sum()),
        "moyenne_chars": round(lengths.mean(), 1) if len(df) else 0,
        "median_chars": int(lengths.median()) if len(df) else 0,
        "min_chars": int(lengths.min()) if len(df) else 0,
        "max_chars": int(lengths.max()) if len(df) else 0,
    })

char_df = pd.DataFrame(char_stats).sort_values("total_chars", ascending=False).reset_index(drop=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)

print("=" * 100)
print("CHARACTER SIZE OF THE 'content' COLUMN PER DATASET (disjoint files)")
print("=" * 100)
print(char_df.to_string(index=False))

# Global total
total_chars_all = char_df["total_chars"].sum()
total_lines_all = char_df["n_lignes"].sum()

print("\n" + "=" * 100)
print(f"GLOBAL TOTAL of characters (content) : {total_chars_all:,}".replace(",", " "))
print(f"GLOBAL TOTAL of rows                 : {total_lines_all:,}".replace(",", " "))
print(f"Global average chars/row             : "
      f"{round(total_chars_all / total_lines_all, 1) if total_lines_all else 0}")

# Save the report
report_path = os.path.join(DATA_DIR, "content_char_stats.csv")
char_df.to_csv(report_path, sep=",", encoding="utf-8", index=False)
print(f"\nReport saved: {report_path}")